# TASK 2 — Data Cleaning (Telco Customer Churn)

**Prepared by: FEDELIKA MAXMUS**

## 1. Project / Task Objective

| Task-list requirement | How we do it in this notebook |
|-----------------------|-------------------------------|
| 1. Handle missing values appropriately (fill, drop, or impute) | Diagnose `isnull()` + hidden blanks, coerce `TotalCharges` to numeric, fill its 11 NaNs with `0` (tenure = 0 logic) |
| 2. Remove duplicate records if any | Check `duplicated().sum()`, check `customerID` uniqueness, run `drop_duplicates()` |
| 3. Standardize column names (lowercase, spaces → underscores) | `str.strip().str.lower().str.replace(' ', '_')` + verification |

**Link to Task 1 findings:** Task 1 found shape `(7043, 21)`, `TotalCharges` stored as `object/string` instead of numeric, **0 nulls by `isnull()` but 11 blank strings in `TotalCharges`** (all with `tenure = 0`), and **0 duplicate rows**. This notebook fixes exactly those issues and saves a clean dataset for Task 3 (EDA).

## 2. Import Libraries

Same stack as Task 1 so Tasks 1–5 stay comparable. We only need `pandas` (+ `numpy` for numeric ops) for cleaning.

In [ ]:
import pandas as pd
import numpy as np

## 3. Load Dataset — BEFORE snapshot

We reload the **raw** CSV and take a "before cleaning" snapshot: shape, columns, dtypes, nulls. This proves what changed after cleaning.

In [ ]:
df = pd.read_csv("Telco_Customer_Churn_Dataset  (3).csv")

print("BEFORE — Shape (rows, columns):", df.shape)
print("\nBEFORE — Columns:")
print(df.columns.tolist())

BEFORE — Shape (rows, columns): (7043, 21)

BEFORE — Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [ ]:
print("--- BEFORE dtypes ---")
print(df.dtypes)
print("\n--- BEFORE nulls (isnull) ---")
print("Total nulls by isnull():", int(df.isnull().sum().sum()), "  <-- misleading, hidden blanks exist")


--- BEFORE dtypes ---
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

--- BEFORE nulls (isnull) ---
Total nulls by isnull(): 0   <-- misleading, hidden blanks exist


## 4. STEP 1 — Standardize Column Names

**Task-list rule:** convert to lowercase and replace spaces with underscores.

**Why it matters (business + code reason):**
- `SeniorCitizen` vs `seniorcitizen` vs `Senior Citizen` causes `KeyError` bugs and breaks merges/groupbys in Tasks 3–5.
- One convention (`snake_case`, lowercase) makes EDA, segmentation and modelling code readable and reproducible.

**Method:** `strip()` (remove accidental leading/trailing spaces) → `lower()` → `replace(' ', '_')`. We also assert no duplicates were created by the rename.

In [ ]:
print("BEFORE:", df.columns.tolist()[:5], "...")

# Task-list rule: lowercase + spaces -> underscores (+ strip for safety)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

print("AFTER :", df.columns.tolist()[:5], "...")
print("Full cleaned column list:")
print(df.columns.tolist())

# Verification: lowercase, no spaces, no duplicate names
assert (df.columns == [c.lower() for c in df.columns]).all(), "Not all lowercase!"
assert not any(" " in c for c in df.columns), "Spaces remain!"
assert not df.columns.duplicated().any(), "Duplicate column names!"
print("Column-name check passed: 21 names, all lowercase, no spaces, no duplicates.")

BEFORE: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents'] ...
AFTER : ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents'] ...
Full cleaned column list:
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']
Column-name check passed: 21 names, all lowercase, no spaces, no duplicates.


## 5. STEP 2a — Diagnose Missing Values (nulls + hidden blanks)

**Key lesson from Task 1:** `df.isnull().sum()` reports **0 missing values**, but that is **misleading**.

This Telco CSV stores some missing values as **empty / whitespace-only strings** (`""` / `" "`), which pandas does NOT count as `NaN`. The known culprit is `totalcharges`.

So we check three layers:
1. Standard `NaN` counts (`isnull()`)
2. Blank-string counts per text column (`str.strip() == ""`)
3. Coerce `totalcharges` to numeric with `pd.to_numeric(errors='coerce')` — blanks become real `NaN` we can count and fix.

In [ ]:
print("--- Standard NaN counts ---")
print("Total NaNs:", int(df.isnull().sum().sum()))

print("\n--- Blank / whitespace-only strings per text column ---")
obj_cols = df.select_dtypes(include=["object", "string"]).columns
blank_counts = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in obj_cols}
# Show only columns with blanks (keeps output clean)
for c, n in blank_counts.items():
    if n > 0:
        print(f"{c}: {n}  <-- only column with hidden missing")
print("(all other text columns: 0 blanks)")

# Convert TotalCharges to numeric: blanks -> NaN, valid strings -> float
df["totalcharges"] = pd.to_numeric(df["totalcharges"].astype(str).str.strip(), errors="coerce")
print("\n--- After pd.to_numeric(totalcharges, errors='coerce') ---")
print("NaNs in totalcharges:", int(df["totalcharges"].isnull().sum()))
print("totalcharges dtype after coerce:", df["totalcharges"].dtype)

--- Standard NaN counts ---
Total NaNs: 0

--- Blank / whitespace-only strings per text column ---


totalcharges: 11  <-- only column with hidden missing
(all other text columns: 0 blanks)

--- After pd.to_numeric(totalcharges, errors='coerce') ---
NaNs in totalcharges: 11
totalcharges dtype after coerce: float64


In [ ]:
# Who are the 11 affected customers? Business context matters for the fix.
mask = df["totalcharges"].isnull()
print("Rows where totalcharges is NaN (11 rows):")
print(df.loc[mask, ["customerid", "tenure", "monthlycharges", "totalcharges", "contract"]].head(3).to_string())
print("... (all 11 have tenure == 0, i.e. brand-new customers)")
print("\nCheck: all 11 NaN rows have tenure == 0 ?", bool((df.loc[mask, "tenure"] == 0).all()))
print(f"Share of dataset: {int(mask.sum())} / {len(df)} = {mask.mean()*100:.2f}%")

Rows where totalcharges is NaN (11 rows):


     customerid  tenure  monthlycharges  totalcharges  contract
488  4472-LVYGI       0           52.55           NaN  Two year
753  3115-CZMZD       0           20.25           NaN  Two year
936  5709-LVOEQ       0           80.85           NaN  Two year
... (all 11 have tenure == 0, i.e. brand-new customers)

Check: all 11 NaN rows have tenure == 0 ? True
Share of dataset: 11 / 7043 = 0.16%


## 6. STEP 2b — Handle Missing Values (decision + fix)

**What we found:** 11 missing `totalcharges`, **all with `tenure = 0`** (just joined, no billing history yet). No other column has missing values.

**Options considered:**

| Option | Value it would put | Verdict |
|--------|-------------------|---------|
| A. Fill with `0` ✅ **chosen** | `0.0` | **Business-correct:** `tenure = 0 months × monthly charge = 0 total`. Keeps all 7043 rows. |
| B. Drop the 11 rows | — | Would work (only 0.16% loss) but throws away valid new customers; worse than A. |
| C. Fill with median (~1397.5) / mean (~2283.3) | ~1397.5 | **Wrong here:** assigns 1–2 years of charges to a 0-month customer; distorts EDA means and churn analysis. |

We therefore **fill with 0** and convert the column to `float64`. This is a *fill* strategy per the task list ("fill, drop, or impute"), with reasoning documented.

In [ ]:
print(f"Median totalcharges (for reference, NOT used): {df['totalcharges'].median():.2f}")
print("NaNs before fill:", int(df["totalcharges"].isnull().sum()))

# Business-logic fix: tenure=0 -> totalcharges should be 0
df["totalcharges"] = df["totalcharges"].fillna(0)

print("NaNs after fill with 0:", int(df["totalcharges"].isnull().sum()))
print("totalcharges dtype:", df["totalcharges"].dtype, "— now numeric, ready for EDA/stats.")

Median totalcharges (for reference, NOT used): 1397.47
NaNs before fill: 11
NaNs after fill with 0: 0
totalcharges dtype: float64 — now numeric, ready for EDA/stats.


## 7. STEP 3 — Remove Duplicates

**Task-list rule:** remove duplicate records if any.

**Why:** duplicate rows double-count customers, inflate churn rates and bias every chart in Tasks 3–5.

**Method:**
1. Count full-row duplicates with `duplicated().sum()`
2. Confirm business key `customerid` is unique (`nunique() == len(df)`)
3. Run `drop_duplicates()` anyway (idempotent — safe even when count is 0) and verify shape is unchanged.

In [ ]:
dups_before = int(df.duplicated().sum())
print("Duplicate full rows (before):", dups_before)
print(f"Unique customerid: {df['customerid'].nunique()} / {len(df)} rows — key is unique, no ID collisions.")

rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after drop_duplicates(): {df.shape} — unchanged, as expected.")
assert len(df) == rows_before - dups_before

Duplicate full rows (before): 0
Unique customerid: 7043 / 7043 rows — key is unique, no ID collisions.


Shape after drop_duplicates(): (7043, 21) — unchanged, as expected.


## 8. Final Hygiene — strip whitespace, verify no residual issues

Extra safety pass before saving:
- `strip()` whitespace from all text columns (e.g. `"Yes "` vs `"Yes"` would split groups in Task 5 demographics).
- Re-confirm: **0 nulls everywhere**, correct dtypes (`totalcharges` is now `float64`), shape still `(7043, 21)`.

In [ ]:
# Strip whitespace from text columns to avoid 'Yes' vs 'Yes ' grouping bugs
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].astype(str).str.strip()

print("--- AFTER dtypes ---")
print(df[["customerid", "gender", "seniorcitizen", "partner", "dependents", "tenure", "monthlycharges", "totalcharges", "churn"]].dtypes)
print("(+ 12 more object columns)")
print("\n--- AFTER nulls ---")
print("Total nulls:", int(df.isnull().sum().sum()), "— PASS" if df.isnull().sum().sum() == 0 else "— FAIL")
print("Duplicate rows:", int(df.duplicated().sum()), "— PASS" if df.duplicated().sum() == 0 else "— FAIL")
print("Final shape:", df.shape, "— no rows lost.")

--- AFTER dtypes ---
customerid            str
gender                str
seniorcitizen       int64
partner               str
dependents            str
tenure              int64
monthlycharges    float64
totalcharges      float64
churn                 str
dtype: object
(+ 12 more object columns)

--- AFTER nulls ---


Total nulls: 0 — PASS
Duplicate rows: 0 — PASS
Final shape: (7043, 21) — no rows lost.


## 9. Save Cleaned Dataset

We save the cleaned frame so **Tasks 3 (EDA), 4 (Segmentation) and 5 (Advanced Analysis)** all start from identical, reproducible input. The raw CSV is never overwritten.

In [ ]:
df.to_csv("Telco_Customer_Churn_Cleaned.csv", index=False)
print(f"Saved: Telco_Customer_Churn_Cleaned.csv ({df.shape[0]} rows x {df.shape[1]} columns)")

# Reload check: proves the file on disk is valid
check = pd.read_csv("Telco_Customer_Churn_Cleaned.csv")
print(f"Reload check — shape: {check.shape}, nulls: {int(check.isnull().sum().sum())}. OK.")

Saved: Telco_Customer_Churn_Cleaned.csv (7043 rows x 21 columns)
Reload check — shape: (7043, 21), nulls: 0. OK.


## 10. Findings / Conclusion

**Before → After:**
- Shape: `(7043, 21)` → `(7043, 21)` — no rows lost (correct: only hidden missing + naming issues existed).
- Columns: `customerID, SeniorCitizen, ...` → `customerid, seniorcitizen, ...` — all 21 lowercase, no spaces, verified unique.
- `totalcharges`: `object` with 11 hidden blanks → `float64` with **0 nulls** (blanks coerced to `NaN`, then filled with `0` because all 11 rows have `tenure = 0`).
- Duplicates: 0 found, `drop_duplicates()` applied, `customerid` confirmed unique (7043/7043).
- Final validation: `isnull().sum() = 0`, `duplicated().sum() = 0`, cleaned CSV saved as `Telco_Customer_Churn_Cleaned.csv`.

**Why fill-with-0 (not median/drop)?** Median (~1397.5) would invent a year of charges for brand-new customers; dropping would discard valid customers for no gain (0.16%). Zero reflects the business reality `0 months tenure = 0 total charges`.
